# Pelatihan Model Segmentasi Luka U-Net Menggunakan PyTorch
### Tugas Akhir / Projek Akhir Sistem Multimedia (UNNES)

Notebook ini memandu Anda dalam melakukan pelatihan model Deep Learning berbasis segmentasi semantik (**U-Net**) menggunakan dataset **CO2Wounds-V2** (Chronic Leprosy Wound Dataset). 

Model yang dihasilkan dari pelatihan ini dapat diekspor ke dalam format **ONNX** dan **TensorFlow.js** untuk dideploy secara *client-side* di aplikasi web atau PWA seluler.

## 1. Persiapan Lingkungan & Instalasi Pustaka
Jika dijalankan di Google Colab, Anda dapat mengaktifkan **GPU T4** (Runtime -> Change runtime type -> T4 GPU) untuk mempercepat proses training.

In [ ]:
# Pastikan GPU aktif
import torch
print("CUDA Tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nama GPU:", torch.cuda.get_device_name(0))

# Install pustaka yang diperlukan untuk ekspor model ke format web
!pip install -q onnx tf2onnx tensorflowjs matplotlib

## 2. Struktur Data & Dataset Loader
Gunakan struktur dataset dari folder proyek Anda. Jika berada di Google Colab, Anda dapat mengompres folder `CO2Wounds-V2...` menjadi format `.zip` dan mengunggahnya ke runtime Colab terlebih dahulu.

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# Konfigurasi Path (Sesuaikan dengan lokasi folder Anda di Google Colab)
DATASET_DIR = "/content/CO2Wounds-V2 Extended Chronic Wounds Dataset From Leprosy Patients"

# Jika file di-zip, gunakan perintah berikut untuk unzip:
# !unzip -q "/content/dataset.zip" -d "/content/"

IMGS_DIR = os.path.join(DATASET_DIR, "imgs")
MASKS_DIR = os.path.join(DATASET_DIR, "masks")

### Pembuatan Custom Dataset & Augmentasi Citra
Augmentasi acak (Horizontal/Vertical Flip & Rotation) digunakan agar model kuat terhadap variasi pengambilan foto menggunakan smartphone (posisi miring, jarak foto, dsb).

In [ ]:
class WoundDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = sorted(image_paths)
        self.mask_paths = sorted(mask_paths)
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")
        
        if self.transform:
            image, mask = self.transform(image, mask)
            
        return image, mask

class JointTransform:
    def __init__(self, size=(256, 256), train=True):
        self.size = size
        self.train = train

    def __call__(self, image, mask):
        # Resize
        image = transforms.Resize(self.size)(image)
        mask = transforms.Resize(self.size, interpolation=transforms.InterpolationMode.NEAREST)(mask)

        if self.train:
            if np.random.rand() > 0.5:
                image = transforms.functional.hflip(image)
                mask = transforms.functional.hflip(mask)
            if np.random.rand() > 0.5:
                image = transforms.functional.vflip(image)
                mask = transforms.functional.vflip(mask)
            # Rotasi acak
            angle = np.random.uniform(-15, 15)
            image = transforms.functional.rotate(image, angle)
            mask = transforms.functional.rotate(mask, angle)

        image = transforms.ToTensor()(image)
        # Normalisasi ImageNet standar
        image = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(image)
        
        mask = transforms.ToTensor()(mask)
        mask = (mask > 0.5).float() # Konversi biner ke 0 atau 1

        return image, mask

## 3. Visualisasi Dataset
Menampilkan citra asli dan masker ground truth secara berdampingan untuk memastikan data ter-load dengan benar.

In [ ]:
all_images = glob.glob(os.path.join(IMGS_DIR, "*"))
image_paths = []
mask_paths = []

for img_path in all_images:
    base_name = os.path.basename(img_path)
    name_without_ext = os.path.splitext(base_name)[0]
    mask_file = os.path.join(MASKS_DIR, f"{name_without_ext}.png")
    if os.path.exists(mask_file):
        image_paths.append(img_path)
        mask_paths.append(mask_file)

print(f"Pasangan data terdeteksi: {len(image_paths)}")

# Tampilkan sampel visual citra & masker kusta kronis
if len(image_paths) > 0:
    plt.figure(figsize=(10, 5))
    idx = np.random.randint(len(image_paths))
    
    plt.subplot(1, 2, 1)
    plt.imshow(Image.open(image_paths[idx]))
    plt.title("Citra Luka Pasien")
    plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.imshow(Image.open(mask_paths[idx]), cmap="gray")
    plt.title("Masker Ground Truth (Luka)")
    plt.axis("off")
    
    plt.show()

## 4. Arsitektur Jaringan U-Net
U-Net terdiri dari jalur kontraksi (Encoder/Downsampling) untuk menangkap konteks spasial, dan jalur ekspansi (Decoder/Upsampling) untuk melokalisasi koordinat piksel luka secara presisi.

In [ ]:
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential( 
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        features = [64, 128, 256, 512]
        current_in = in_channels
        for feature in features:
            self.downs.append(DoubleConv(current_in, feature))
            current_in = feature
            
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)
        
        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feature * 2, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature * 2, feature))
            
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)
            
        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]
        
        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x) 
            skip_connection = skip_connections[idx // 2]
            concat_x = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx + 1](concat_x)
            
        return torch.sigmoid(self.final_conv(x))

## 5. Pelatihan Model (Training Loop)
Menggunakan kombinasi hybrid **BCE Loss** (untuk klasifikasi piksel biner) dan **Dice Loss** (untuk memaksimalkan nilai overlap area luka).

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self): super(DiceBCELoss, self).__init__()
    def forward(self, inputs, targets, smooth=1):
        bce_loss = nn.BCELoss()(inputs, targets)
        inputs_flat = inputs.view(-1)
        targets_flat = targets.view(-1)
        intersection = (inputs_flat * targets_flat).sum()                            
        dice_loss = 1 - ((2. * intersection + smooth) / (inputs_flat.sum() + targets_flat.sum() + smooth))  
        return bce_loss + dice_loss

# Split Data 80% Train, 20% Val
indices = np.arange(len(image_paths))
np.random.seed(42)
np.random.shuffle(indices)
split_idx = int(len(image_paths) * 0.8)

train_loader = DataLoader(
    WoundDataset([image_paths[i] for i in indices[:split_idx]], [mask_paths[i] for i in indices[:split_idx]], transform=JointTransform(train=True)),
    batch_size=8, shuffle=True
)
val_loader = DataLoader(
    WoundDataset([image_paths[i] for i in indices[split_idx:]], [mask_paths[i] for i in indices[split_idx:]], transform=JointTransform(train=False)),
    batch_size=8, shuffle=False
)

# Inisialisasi parameter training
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet().to(DEVICE)
criterion = DiceBCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# Training loop
epochs = 15
print("Memulai training...")
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = model(imgs)
        loss = criterion(preds, masks)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Loss: {train_loss/len(train_loader):.4f}")

# Simpan Bobot Model
torch.save(model.state_dict(), "wound_unet.pth")
print("Model disimpan di wound_unet.pth")

## 6. Visualisasi Hasil Prediksi Deteksi
Menampilkan visual perbandingan tiga kolom: **Citra Asli**, **Masker Sebenarnya**, dan **Hasil Prediksi AI**.

In [ ]:
model.eval()
imgs, masks = next(iter(val_loader))
with torch.no_grad():
    preds = model(imgs.to(DEVICE))
    
# Plot sampel prediksi
plt.figure(figsize=(12, 4))
for i in range(3):
    plt.subplot(3, 3, i*3 + 1)
    # Denormalisasi citra untuk visualisasi matplotlib
    img_vis = imgs[i].permute(1, 2, 0).numpy()
    img_vis = img_vis * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    plt.imshow(np.clip(img_vis, 0, 1))
    plt.title("Foto Asli")
    plt.axis("off")
    
    plt.subplot(3, 3, i*3 + 2)
    plt.imshow(masks[i].squeeze(), cmap="gray")
    plt.title("Anotasi Asli")
    plt.axis("off")
    
    plt.subplot(3, 3, i*3 + 3)
    pred_vis = (preds[i].cpu().squeeze() > 0.5).float()
    plt.imshow(pred_vis, cmap="teal")
    plt.title("Prediksi AI (Segmentasi)")
    plt.axis("off")
    
plt.tight_layout()
plt.show()

## 7. Ekspor ke ONNX & TensorFlow.js (Konversi untuk Web/PWA)
Pertama, kita ekspor model PyTorch ke format perantara standard ONNX.

In [ ]:
dummy_input = torch.randn(1, 3, 256, 256, device=DEVICE)
torch.onnx.export(
    model, dummy_input, "wound_unet.onnx",
    export_params=True, opset_version=11,
    input_names=["input"], output_names=["output"]
)
print("Model berhasil diekspor ke wound_unet.onnx")

### Konversi ONNX ke TensorFlow.js GraphModel
Jalankan baris perintah di bawah untuk mengubah file ONNX menjadi folder model TensorFlow.js yang siap diimpor di aplikasi web secara offline menggunakan JavaScript.

In [ ]:
# Jalankan tool konversi tensorflowjs
!tensorflowjs_converter --input_format=onnx --output_node_names='output' wound_unet.onnx ./model_tfjs

print("Konversi selesai! Silakan unduh folder 'model_tfjs' dan masukkan ke dalam aset web Anda.")